# Profiling VMC Runs

This notebook profiles the actual 1D TFIM ground-state search demo used elsewhere in this repository.

- Use `cProfile` when you want Python-level hotspots.
- Use `snakeviz` to inspect a saved `cProfile` result visually.
- Use `scalene` when you want Python/native time and memory breakdowns.
- Use `jax.profiler` when the slow part is likely inside JAX/XLA execution.

The workload below runs the full `ising1d_ed_vs_vmc` demo, so the profile reflects the real RBM/FFNN/CNN ground-state search path rather than a toy VMC loop.

In [ ]:
from pathlib import Path

from performance_profiling_helper import (
    cprofile_groundstate_demo_run,
    scalene_command,
    tensorboard_command,
    trace_groundstate_demo_with_jax_profiler,
)

## Python-level profiling with cProfile

This is the first tool to use when the overhead might come from driver logic, repeated object construction, sampling, or demo orchestration.

In [ ]:
stats_path = Path("profiling_results") / "cprofile_groundstate_demo.prof"
profile_result = cprofile_groundstate_demo_run(length=5, transverse_field=1.0, output_path=stats_path)
print(profile_result["table"])
print()
print(profile_result["summary"])
print(f"Saved cProfile stats to: {stats_path.resolve()}")

If you want a visual browser for the saved `cProfile` output, run this in a terminal:

```powershell
snakeviz Balint/demos/profiling_results/cprofile_groundstate_demo.prof
```

## JAX tracing

Use this when `cProfile` shows little Python time but the run is still slow. That usually means the cost is in JAX/XLA execution rather than pure Python.

The default trace below profiles only the CNN branch of the ground-state search demo with a shorter profiling budget. Profiling the full RBM+FFNN+CNN demo under JAX tracing is possible, but it is much slower.

In [ ]:
trace_dir = Path("profiling_results") / "jax_profile_groundstate_demo"
result = trace_groundstate_demo_with_jax_profiler(
    trace_dir,
    model_name="CNN",
    length=5,
    transverse_field=1.0,
    fast_profile=True,
)
print(f"Wrote JAX trace to: {trace_dir.resolve()}")
print(tensorboard_command(trace_dir.resolve()))
for entry in result["results"]:
    print(entry["model"], entry["final_energy"])

Open the trace with TensorBoard using the printed command, then open the local URL it shows and go to the **Profile** tab.

If you really need the full demo trace, call `trace_groundstate_demo_with_jax_profiler(..., model_name=None, fast_profile=False)`, but expect it to take much longer.

## Scalene

Scalene runs from the terminal. The helper prints a ready-to-run command.

In [ ]:
print(scalene_command())